In [3]:
import json
from wsgiref import headers

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from bs4 import BeautifulSoup
import gradio as gr

print("Loading environment variables...")
load_dotenv(override=True)

requests.get("http://localhost:11434").content
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_links(url):
    """
    Returns the links on the website on the given url
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


links = fetch_website_links("https://edwarddonner.com")
links



j:\Jayashree\AI_engineer_projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading environment variables...


['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

"""
    
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    print(user_prompt)
    return user_prompt

In [17]:
print(get_links_user_prompt("https://edwarddonner.com"))


You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder

In [6]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} ")
    response = ollama.chat.completions.create(
        model="gpt-oss:20b",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [9]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com 

You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company information',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'}]}

In [7]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [8]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

Selecting relevant links for https://edwarddonner.com 

You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic

In [9]:
brochure_system_prompt = """
You are a senior marketing copywriter.
Write a polished company brochure in Markdown that reads like a real brochure, not a summary.
Use the provided company information to create a concise, professional brochure with sections such as:
- Company Overview
- Why Choose Us
- Customers and Culture
- Careers and Opportunities
Do not ask follow-up questions. Do not include code blocks. Do not write a plain summary.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# brochure_system_prompt = """
# You are a marketing copywriter. Read the page content and links, then
# write a short brochure in Markdown. Do not ask any follow-up questions.


In [10]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called {company_name} Here are its contents of its landing page and other relevant pages.
Use this information to build a short brochure of the company in markdown without code blocks. \n """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com 

You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic

'\nYou are looking at a company called Edward Donner Here are its contents of its landing page and other relevant pages.\nUse this information to build a short brochure of the company in markdown without code blocks. \n ## Landing Page:\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a Managing Director at JPMorgan.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, a

In [11]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2:latest",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [12]:
create_brochure("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com 

You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic

# Edward Donner
A New Paradigm for AI-Driven Human Prosperity

As the founder and CTO of Nebula.io, we're revolutionizing the way recruiters source, understand, engage, and manage talent using Generative AI and machine learning. Our patented model matches people with roles with unprecedented accuracy and speed – no keywords required.

### Our Vision

At Edward Donner, we believe that discovering one's potential and pursuing their reason for being is essential to human prosperity. Our long-term goal is to help individuals find roles where they'll be most fulfilled and successful, leading to a higher level of prosperity for all.

### About Us

I'm Ed, the co-founder and CTO of Nebula.io. I've had the privilege of working with top organizations like JPMorgan and untapt (acquired in 2021), and I'm now dedicated to finding people their dream jobs using AI. When I'm not working, you can find me producing amateur electronic music or discussing the latest advancements in LLMs and Agents – a passion that inspired our Udemy courses on Coder, Builder, and Agentic Engineer.

### Our Arena

Outsmart is an arena where Large Language Models (LLMs) compete against each other in a battle of diplomacy and deviousness. We're fascinated by the potential of these models to revolutionize industries and solve real-world problems – including hiring and talent management.

### Get in Touch

Ready to discover your potential and find inspiration at work? Stay up-to-date with our latest news, resources, and courses on AI Coder, Builder, Agentic Engineer, and more. Reach out to me directly to chat about all things LLMs and Agents.

Follow me:

* LinkedIn: [Linkedin Profile]
* Twitter: [Twitter Handle]
* Facebook: [Facebook Page]

In [13]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=create_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co"],
            ["Edward Donner", "https://edwarddonner.com"],
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://edwarddonner.com 

You are provided with the list of links on a website. You are able to find out which of the links are related to
include in a brochure for the company, 
respond with a full url in a JSON format
Donot include Terms of Service, Privacy Policy, or any other unrelated links in the response.

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic

**Edward Donner: Revolutionizing the Future of Work**

Welcome to Edward Donner, an AI-powered innovation that's changing the way we approach talent acquisition. Our mission is to help people discover their potential and pursue their reason for being, driven by a concept called Ikigai.

**Our Story**

Founded by Ed, our co-founder and CTO, Nebula.io was born from a passion to apply AI to real-world problems. After years of experience in the field, Ed saw firsthand the complexity of hiring and the need for a more efficient solution. Our patented model matches people with roles with greater accuracy and speed than previously imaginable – no keywords required. We're proud to say that our platform is trusted by recruiting teams at leading organizations nationwide.

**What Sets Us Apart**

At Edward Donner, we're not just building an AI-powered talent acquisition platform; we're redefining the future of work. Our platform unifies candidate sourcing, outreach automation, and talent intelligence into one seamless recruiting workflow. With Nebula, you can:

* Find better candidates
* Automate recruiter workflows
* Make faster hiring decisions

**Join Our Community**

We believe that innovation should be accessible to everyone. That's why we offer a range of resources, including best-selling Udemy courses on LLMs and Agents, as well as live events and talks with expert Ed himself.

**Stay in Touch**

Follow us on LinkedIn, Twitter, or Facebook to stay up-to-date on the latest news and innovations from Edward Donner. Contact us at [ed[at]edwarddonner.com](mailto:ed[at]edwarddonner.com) to learn more about how we can help you revolutionize your recruitment process.

**Join the Movement**

Let's work together to make hiring faster, smarter, and more fulfilling for everyone involved.